# Arabic Handwritten OCR Pipeline
This notebook contains the complete pipeline for detecting, recognizing, and auto-labeling Arabic handwritten characters and words.

In [9]:
import os
import cv2
import numpy as np
import pandas as pd
import pathlib
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tqdm import tqdm
import matplotlib.pyplot as plt
import kagglehub

# AHCD Alphabet Mapping (Standard Arabic Order)
ARABIC_ALPHABET = ['أ', 'ب', 'ت', 'ث', 'ج', 'ح', 'خ', 'د', 'ذ', 'ر', 'ز', 'س', 'ش', 'ص', 'ض', 'ط', 'ظ', 'ع', 'غ', 'ف', 'ق', 'ك', 'ل', 'م', 'ن', 'هـ', 'و', 'ي']

## 1. Load Pre-training Data (AHCD)
We use the 16,800 images from the Arabic Handwritten Characters Dataset to build a robust foundation.

In [10]:
path = kagglehub.dataset_download("mloey1/ahcd1")
train_images_path = os.path.join(path, 'csvTrainImages 13440x1024.csv')
train_labels_path = os.path.join(path, 'csvTrainLabel 13440x1.csv')

X_train_large = pd.read_csv(train_images_path, header=None).values.reshape(-1, 32, 32, 1).astype('float32') / 255.0
y_train_large = pd.read_csv(train_labels_path, header=None).values.flatten() - 1

# Correct orientation for the standard AHCD format
X_train_large = np.rot90(X_train_large, k=-1, axes=(1, 2))
X_train_large = np.flip(X_train_large, axis=2)

print(f'Loaded {len(X_train_large)} training samples.')

Loaded 13440 training samples.


## 2. Build and Train the 'Arabic Expert' Model

In [11]:
def build_expert_model():
    model = models.Sequential([
        layers.Input(shape=(32, 32, 1)),
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(28, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

model_arabic_expert = build_expert_model()
model_arabic_expert.fit(X_train_large, y_train_large, epochs=10, batch_size=64, validation_split=0.1)

Epoch 1/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 12s 52ms/step - accuracy: 0.5728 - loss: 1.4681 - val_accuracy: 0.0357 - val_loss: 6.9059
Epoch 2/10
 25/189 ━━━━━━━━━━━━━━━━━━━━ 7s 47ms/step - accuracy: 0.8273 - loss: 0.5490

KeyboardInterrupt: 

## 3. Recognition Logic (Word & Batch Processing)

In [12]:
def predict_word(image_path, boxes):
    rtl_boxes = sorted(boxes, key=lambda b: -b[0])
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    batch = []
    for (x, y, w, h) in rtl_boxes:
        roi = cv2.resize(img[y:y+h, x:x+w], (32, 32))
        batch.append(roi.reshape(32, 32, 1).astype('float32') / 255.0)

    preds = model_arabic_expert.predict(np.array(batch), verbose=0)
    return "".join([ARABIC_ALPHABET[i] for i in np.argmax(preds, axis=1)])

def batch_auto_label_recursive(root_folder, confidence_threshold=0.95):
    all_files = list(pathlib.Path(root_folder).rglob('*.[pj][np][ge]*'))
    results = []
    for i in tqdm(range(0, len(all_files), 128)):
        batch_files = all_files[i : i + 128]
        imgs = []
        paths = []
        for p in batch_files:
            img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
            if img is not None:
                imgs.append(cv2.resize(img, (32, 32)).reshape(32, 32, 1).astype('float32') / 255.0)
                paths.append(str(p))

        if not imgs: continue
        preds = model_arabic_expert.predict(np.array(imgs), verbose=0)
        for idx, p_val in enumerate(preds):
            if np.max(p_val) >= confidence_threshold:
                results.append({'path': paths[idx], 'char': ARABIC_ALPHABET[np.argmax(p_val)], 'conf': np.max(p_val)})

    df = pd.DataFrame(results)
    df.to_csv('ocr_results.csv', index=False)
    return df

### Pixel-Based Letter Detection
This approach uses image processing to automatically find black letters by clustering dark pixels.

In [13]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

def mark_every_pixel(image_path):
    img = cv2.imread(image_path)
    if img is None:
        print(f'Error: Could not load image at {image_path}')
        return

    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Find coordinates of all black pixels (threshold < 100)
    y_coords, x_coords = np.where(gray < 100)

    # Create a white background to clearly see the 'pixel' annotations
    output_img = np.full((gray.shape[0], gray.shape[1], 3), 255, dtype=np.uint8)

    # Draw a red dot for every single detected pixel
    for x, y in zip(x_coords, y_coords):
        output_img[y, x] = [255, 0, 0]

    plt.figure(figsize=(16, 16))
    plt.imshow(output_img)
    plt.title(f'Visualizing {len(x_coords)} Individual Black Pixels')
    plt.axis('off')
    plt.show()

mark_every_pixel('/content/LETTERS.png')

Error: Could not load image at /content/LETTERS.png


### Preparing the Arabic Handwritten Dataset
Now that we can detect the letter regions (clusters of pixels), the next step is to assign the correct Arabic character to each detection. This will create the ground truth labels for your ML model.

In [14]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

def extract_and_label_letters(image_path):
    img = cv2.imread(image_path)
    if img is None: return [], []
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Moderate threshold to balance sensitivity and noise
    _, thresh = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY_INV)

    # Smaller kernel to prevent over-merging adjacent letters
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    dilated = cv2.dilate(thresh, kernel, iterations=1)

    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Filter out very small dots that are likely noise (area < 10)
    bounding_boxes = [cv2.boundingRect(c) for c in contours if cv2.contourArea(c) >= 10]

    # Sort: Top-to-Bottom, then Right-to-Left within lines
    line_tolerance = 60
    bounding_boxes = sorted(bounding_boxes, key=lambda b: (b[1] // line_tolerance, -b[0]))

    labeled_dataset = []
    for i, (x, y, w, h) in enumerate(bounding_boxes):
        letter_roi = gray[y:y+h, x:x+w]
        resized = cv2.resize(letter_roi, (32, 32), interpolation=cv2.INTER_AREA)
        labeled_dataset.append({
            'id': i + 1,
            'image': resized.tolist(),
            'label': None
        })

    return labeled_dataset, bounding_boxes

dataset, final_boxes = extract_and_label_letters('/content/LETTERS.png')
print(f'Balanced update: Now detected {len(dataset)} regions.')

Balanced update: Now detected 0 regions.


In [15]:
import cv2
import matplotlib.pyplot as plt

def show_full_image_with_ids(image_path, bounding_boxes):
    img = cv2.imread(image_path)
    if img is None:
        print('Error: Could not load image')
        return
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(20, 25))
    plt.imshow(img_rgb)

    for i, (x, y, w, h) in enumerate(bounding_boxes):
        # Draw ID number
        plt.text(x, y-5, str(i+1), color='red', fontsize=10, fontweight='bold')
        # Draw Bounding Box
        rect = plt.Rectangle((x, y), w, h, fill=False, edgecolor='red', linewidth=1)
        plt.gca().add_patch(rect)

    plt.axis('off')
    plt.title(f'Reference Map: {len(bounding_boxes)} Detections')
    plt.show()

show_full_image_with_ids('/content/LETTERS.png', final_boxes)

Error: Could not load image


### Labeling Arabic Letter Forms
Since Arabic letters have 4 different forms (Isolated, Initial, Medial, Final), we will use a structured labeling system. Use the helper below to inspect and label each detection.

In [16]:
def show_full_image_with_ids(image_path, bounding_boxes):
    img = cv2.imread(image_path)
    if img is None:
        print(f'Error: Could not load image')
        return
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Set figure to align elements clearly
    plt.figure(figsize=(20, 25))
    ax = plt.gca()
    
    plt.imshow(img_rgb)

    for i, (x, y, w, h) in enumerate(bounding_boxes):
        # Use coordinates directly (offsets are already handled in the data creation cell)
        plt.text(x, y-5, str(i+1), color='red', fontsize=10, fontweight='bold', horizontalalignment='left')
        rect = plt.Rectangle((x, y), w, h, fill=False, edgecolor='red', linewidth=1)
        ax.add_patch(rect)

    plt.axis('off')
    plt.title(f'Corrected Reference Map: {len(bounding_boxes)} Detections', loc='left')
    plt.tight_layout()
    plt.show()

show_full_image_with_ids('/content/LETTERS.png', final_boxes)

Error: Could not load image


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import matplotlib.pyplot as plt

if 'refined_labels' not in globals():
    refined_labels = {}

for item in dataset:
    bid = item['id']
    if bid not in refined_labels:
        refined_labels[bid] = {'char': '', 'form': 'Isolated'}

def launch_batch_labeler(dataset):
    output = widgets.Output()
    progress = widgets.IntProgress(value=0, min=0, max=len(dataset), description='Progress:', bar_style='info')
    page_size = 10
    current_page = [0]

    def update_progress():
        progress.value = sum(1 for v in refined_labels.values() if v.get('char') != '')

    def save_batch(controls):
        if not controls: return
        for bid, (char_w, form_w) in controls.items():
            refined_labels[bid] = {'char': char_w.value.strip(), 'form': form_w.value}
        update_progress()

    def show_page(page_idx):
        start = page_idx * page_size
        end = min(start + page_size, len(dataset))

        with output:
            clear_output(wait=True)
            grid_items = []
            current_controls = {}

            if start < len(dataset):
                for i in range(start, end):
                    item = dataset[i]
                    bid = item['id']
                    img = np.array(item['image'])

                    img_out = widgets.Output()
                    with img_out:
                        plt.figure(figsize=(1.2, 1.2))
                        plt.imshow(img, cmap='gray')
                        plt.axis('off')
                        plt.title(f"ID {bid}", fontsize=9)
                        plt.show()

                    val = refined_labels.get(bid, {'char': '', 'form': 'Isolated'})
                    char_in = widgets.Text(value=val['char'], placeholder='Char', layout=widgets.Layout(width='70px'))
                    form_in = widgets.Dropdown(options=['Isolated', 'Initial', 'Medial', 'Final', 'Noise'], value=val['form'], layout=widgets.Layout(width='90px'))

                    current_controls[bid] = (char_in, form_in)
                    grid_items.append(widgets.VBox([img_out, char_in, form_in], layout=widgets.Layout(border='1px solid #ccc', padding='10px', margin='5px')))

                display(widgets.HBox(grid_items, layout=widgets.Layout(flex_flow='row wrap')))
            else:
                print("All items displayed. Click 'Submit & Export' to finish.")

            return current_controls

    # Initialize page
    active_controls = show_page(0)

    # Navigation logic
    def on_prev(b):
        save_batch(active_controls)
        current_page[0] = max(0, current_page[0] - 1)
        active_controls.update(show_page(current_page[0]))

    def on_next(b):
        save_batch(active_controls)
        if (current_page[0] + 1) * page_size < len(dataset):
            current_page[0] += 1
            active_controls.update(show_page(current_page[0]))

    def on_submit(b):
        save_batch(active_controls)
        export_dataset(dataset, refined_labels)

    prev_btn = widgets.Button(description="Prev Page")
    next_btn = widgets.Button(description="Next Page")
    submit_btn = widgets.Button(description="Submit & Export", button_style='danger')

    prev_btn.on_click(on_prev)
    next_btn.on_click(on_next)
    submit_btn.on_click(on_submit)

    nav_box = widgets.HBox([prev_btn, next_btn, submit_btn], layout=widgets.Layout(margin='10px 0'))

    display(progress)
    display(output)
    display(nav_box)

launch_batch_labeler(dataset)

### ✍️ Start Annotating Your Dataset
Use the tool below to map the IDs from the image above to actual Arabic letters.

### Exporting the Dataset
Once you have finished labeling all characters in the `refined_labels` dictionary, run the cell below to save the dataset (images and labels) as a `.npz` file for model training.

### 🖱️ Click-to-Label Interactive Tool
Click on any red box in the image to label that specific character. Finished labels will turn green.

### ➕ Manual Box Creator
If a letter was missed by the automated detector, use this tool to manually add a bounding box. Enter the top-left (x, y) coordinates and the width/height.

### 🖱️ Click-to-Add Missing Letters
Click anywhere on the image below to automatically create a bounding box at that location.

In [ ]:
import json
import cv2
import numpy as np

# Updated annotations for missed letters
new_annotations_json = """{
    \"boxes\": [
        {\"id\": \"3Z\", \"x\": \"425.5\", \"y\": \"575.5\", \"width\": \"33\", \"height\": \"21\"},
        {\"id\": \"3b\", \"x\": \"296.5\", \"y\": \"575.5\", \"width\": \"33\", \"height\": \"21\"},
        {\"id\": \"3d\", \"x\": \"199.5\", \"y\": \"574\", \"width\": \"21\", \"height\": \"22\"},
        {\"id\": \"3f\", \"x\": \"70.5\", \"y\": \"576\", \"width\": \"23\", \"height\": \"18\"}
    ]
}"""

# Prevent doubling boxes if cell is re-run
if 'dataset' in globals() and len(dataset) > 138:
    dataset = dataset[:138]
    final_boxes = final_boxes[:138]

new_data = json.loads(new_annotations_json)
img = cv2.imread('/content/LETTERS.png')
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# User requested shift 'up and to the left' only for these new ones
offset_x = 10
offset_y = 10

for box in new_data['boxes']:
    x = int(float(box['x'])) - offset_x
    y = int(float(box['y'])) - offset_y
    w = int(float(box['width']))
    h = int(float(box['height']))

    final_boxes.append((max(0, x), max(0, y), w, h))

    letter_roi = gray[y:y+h, x:x+w]
    if letter_roi.size > 0:
        resized = cv2.resize(letter_roi, (32, 32), interpolation=cv2.INTER_AREA)
        new_id = len(dataset) + 1
        dataset.append({
            'id': new_id,
            'image': resized.tolist(),
            'label': None
        })

print(f"Corrected: Added {len(new_data['boxes'])} adjusted regions (offset 10). Total dataset size: {len(dataset)}.")

error: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cv::cvtColor'


### ➕ Manual Box Creator
If a letter was missed, enter its coordinates here to add it to the detection list.

### 💾 Export the Labeled Dataset
After you have finished labeling all characters using the batch tool above, run this cell to compile the images and labels into a training-ready format.

In [ ]:
import numpy as np

def export_dataset(dataset, labels_dict):
    images = []
    labels = []
    forms = []

    for item in dataset:
        idx = item['id']
        if idx in labels_dict:
            label_info = labels_dict[idx]
            # Only include if it's not marked as Noise
            if label_info['form'] != 'Noise':
                images.append(np.array(item['image'], dtype=np.uint8))
                labels.append(label_info['char'])
                forms.append(label_info['form'])

    if not images:
        print("No labeled data found to export. Please save some labels first!")
        return

    # Save as compressed numpy array
    np.savez_compressed('arabic_handwritten_dataset.npz',
                        images=np.array(images),
                        char_labels=np.array(labels),
                        form_labels=np.array(forms))

    print(f"Successfully exported {len(images)} characters to 'arabic_handwritten_dataset.npz'!")

# Run this when labeling is complete
export_dataset(dataset, refined_labels)

Successfully exported 138 characters to 'arabic_handwritten_dataset.npz'!


### 📊 Labeling Summary & Final Export
Run the cell below to see a breakdown of your progress and perform the final save.

In [ ]:
import collections

# Calculate counts
stats = collections.Counter([v['form'] for v in refined_labels.values() if v['char'] != ''])
char_counts = collections.Counter([v['char'] for v in refined_labels.values() if v['char'] != ''])

print("--- Dataset Statistics ---")
print(f"Total Labeled: {sum(stats.values())}")
for form, count in stats.items():
    print(f" - {form}: {count}")

print(f"\nUnique Characters: {len(char_counts)}")

# Final Export Trigger
export_dataset(dataset, refined_labels)

--- Dataset Statistics ---
Total Labeled: 0

Unique Characters: 0
Successfully exported 138 characters to 'arabic_handwritten_dataset.npz'!


### 🧪 Synthetic Data Generation
To combat the small sample size, we will use `ImageDataGenerator` to create 10 synthetic variations for every 1 original labeled image. This will help the model see more patterns.

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Load original data
data = np.load('arabic_handwritten_dataset.npz')
orig_images = data['images'].reshape(-1, 32, 32, 1)
orig_chars = data['char_labels']
orig_forms = data['form_labels']

datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.15,
    fill_mode='nearest'
)

synthetic_images = []
synthetic_chars = []
synthetic_forms = []

# Generate 10 variations for each sample
multiplier = 10
for i in range(len(orig_images)):
    img = orig_images[i].reshape((1, 32, 32, 1))
    char = orig_chars[i]
    form = orig_forms[i]

    # Add original
    synthetic_images.append(orig_images[i])
    synthetic_chars.append(char)
    synthetic_forms.append(form)

    # Add augmented
    it = datagen.flow(img, batch_size=1)
    for _ in range(multiplier - 1):
        # FIX: Use next(it) instead of it.next()
        batch = next(it)
        synthetic_images.append(batch[0])
        synthetic_chars.append(char)
        synthetic_forms.append(form)

# Convert to arrays
synthetic_images = np.array(synthetic_images)
synthetic_chars = np.array(synthetic_chars)
synthetic_forms = np.array(synthetic_forms)

# Save the expanded dataset
np.savez_compressed('arabic_augmented_dataset.npz',
                    images=synthetic_images,
                    char_labels=synthetic_chars,
                    form_labels=synthetic_forms)

print(f"Successfully expanded dataset from {len(orig_images)} to {len(synthetic_images)} samples.")

### 🌐 Transfer Learning for Arabic Recognition
Since we have a small dataset, we will use a pre-trained model (MobileNetV2) and fine-tune it. We'll load the augmented data we just created.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 1. Load the Augmented Data
data = np.load('arabic_augmented_dataset.npz')
X = data['images']
y = data['char_labels']

# MobileNetV2 expects 3 channels (RGB) and at least 32x32 pixels.
# We will convert our grayscale images to RGB by repeating the channel.
X_rgb = np.repeat(X.reshape(-1, 32, 32, 1), 3, axis=-1).astype('float32') / 255.0

# 2. Encode Labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)
num_classes = len(le.classes_)

# 3. Split Data
X_train, X_test, y_train, y_test = train_test_split(X_rgb, y_encoded, test_size=0.15, random_state=42)

# 4. Build Transfer Learning Model
base_model = tf.keras.applications.MobileNetV2(input_shape=(32, 32, 3), include_top=False, weights='imagenet')
base_model.trainable = False # Freeze the pre-trained weights

model_tl = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
])

model_tl.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 5. Train
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

print(f"Training Transfer Learning model on {len(X_train)} augmented samples...")
history_tl = model_tl.fit(
    X_train, y_train,
    epochs=50,
    validation_data=(X_test, y_test),
    callbacks=[early_stop],
    verbose=1
)

# Update global variables for evaluation cells
model = model_tl
X_test_eval = X_test
y_test_eval = y_test

### ⚙️ Fine-Tuning the Model
Now that the top layers are trained, we will unfreeze the last 20 layers of the MobileNetV2 base and re-train with a very small learning rate to avoid destroying the pre-trained features.

In [ ]:
# 1. Unfreeze the base model
base_model.trainable = True

# Let's freeze all layers except the last 20
for layer in base_model.layers[:-20]:
    layer.trainable = False

# 2. Re-compile with a much lower learning rate
model_tl.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 3. Fine-tune
print("Fine-tuning the model...")
history_fine = model_tl.fit(
    X_train, y_train,
    epochs=30,
    validation_data=(X_test, y_test),
    callbacks=[early_stop],
    verbose=1
)

Now that we have 1,000 samples, you should re-run the training cell (modifying it to load `arabic_augmented_dataset.npz`) for much better results.

### 🤖 Model Training with Data Augmentation
We will now load the exported dataset, encode the labels, and use Keras Image Augmentation to expand the variety of our training data.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 1. Load Data
data = np.load('arabic_handwritten_dataset.npz')
X = data['images']
y = data['char_labels']

# Reshape and normalize
X = X.reshape(-1, 32, 32, 1).astype('float32') / 255.0

# 2. Encode Labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)
num_classes = len(le.classes_)

# 3. Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.1, random_state=42)

# 4. Data Augmentation
data_augmentation = models.Sequential([
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomTranslation(0.1, 0.1),
])

# 5. Build CNN with Regularization
model = models.Sequential([
    layers.Input(shape=(32, 32, 1)),
    data_augmentation,
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01)),
    layers.Dropout(0.6),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# 6. Define Early Stopping to prevent overfitting
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

print(f"Training with Early Stopping on {len(X_train)} samples.")

history = model.fit(
    X_train, y_train,
    epochs=150,
    validation_data=(X_test, y_test),
    callbacks=[early_stop],
    verbose=1
)

### 📊 Model Evaluation
We will visualize the training process and check the model's precision and recall for each Arabic character.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import numpy as np

# 1. Plot training & validation accuracy/loss
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history_tl.history['accuracy'], label='Initial Train')
if 'history_fine' in globals():
    plt.plot(history_fine.history['accuracy'], label='Fine-tune Train')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_tl.history['loss'], label='Initial Loss')
if 'history_fine' in globals():
    plt.plot(history_fine.history['loss'], label='Fine-tune Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

# 2. Detailed Classification Report
# Using X_test_eval which contains the RGB images
y_pred = np.argmax(model.predict(X_test_eval), axis=1)
unique_test_labels = np.unique(y_test_eval)
target_names_subset = [le.classes_[i] for i in unique_test_labels]

print("\nClassification Report (Test Set):")
print(classification_report(y_test_eval, y_pred, labels=unique_test_labels, target_names=target_names_subset))

# 3. Confusion Matrix Visualization
cm = confusion_matrix(y_test_eval, y_pred, labels=unique_test_labels)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=target_names_subset, yticklabels=target_names_subset, cmap='Blues')
plt.title('Confusion Matrix: Arabic Characters')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

### 📚 Training on a Large Arabic Dataset (AHCD)
Since our initial 100 samples are too few for the model to learn the nuances of Arabic script, we will use the **Arabic Handwritten Characters Dataset** (16,800 images) to provide the model with a strong foundation in Arabic letter forms.

In [ ]:
import kagglehub
import os
import pandas as pd
import numpy as np

# Download AHCD dataset from Kaggle
path = kagglehub.dataset_download("mloey1/ahcd1")
print("Path to dataset files:", path)

# Find the CSV files in the downloaded path
train_images_path = os.path.join(path, 'csvTrainImages 13440x1024.csv')
train_labels_path = os.path.join(path, 'csvTrainLabel 13440x1.csv')

if os.path.exists(train_images_path):
    # Load and preprocess
    X_train_large = pd.read_csv(train_images_path, header=None).values
    y_train_large = pd.read_csv(train_labels_path, header=None).values.flatten() - 1

    X_train_large = X_train_large.reshape(-1, 32, 32, 1).astype('float32') / 255.0
    # Fix orientation to match your LETTERS.png samples
    X_train_large = np.rot90(X_train_large, k=-1, axes=(1, 2))
    X_train_large = np.flip(X_train_large, axis=2)

    print(f'Successfully loaded {len(X_train_large)} professional samples from Kaggle storage.')
else:
    print('CSV files not found in the Kaggle download path. Please check the file structure in:', os.listdir(path))

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# 1. Define a robust CNN architecture for the large dataset
model_arabic_expert = models.Sequential([
    layers.Input(shape=(32, 32, 1)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.GlobalAveragePooling2D(),

    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(28, activation='softmax') # 28 Arabic letters
])

model_arabic_expert.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 2. Train the model
if 'X_train_large' in globals():
    print("Pre-training the 'Arabic Expert' model on AHCD dataset...")
    # Add early stopping to save time if it converges quickly
    early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    history_expert = model_arabic_expert.fit(
        X_train_large, y_train_large,
        epochs=15,
        batch_size=64,
        validation_split=0.1,
        callbacks=[early_stop]
    )
    print("\nPre-training complete!")
else:
    print("Error: X_train_large not found. Please ensure the Kaggle download cell ran successfully.")

### 🔮 Inference: Test on New Images
You can use the function below to predict a character from a cropped image or a file.

In [ ]:
def predict_character(image_array):
    # Ensure image is (1, 32, 32, 1) and normalized
    img = np.array(image_array).reshape(1, 32, 32, 1).astype('float32') / 255.0
    prediction = model.predict(img, verbose=0)
    class_idx = np.argmax(prediction)
    confidence = np.max(prediction)
    return le.classes_[class_idx], confidence

# Example usage on a random test sample
sample_idx = 0
char, conf = predict_character(X_test[sample_idx] * 255)
print(f"Predicted Character: {char} (Confidence: {conf:.2f})")
print(f"Actual Character: {le.inverse_transform([y_test[sample_idx]])[0]}")

### 📖 Word Recognition (Sequence Prediction)
This function takes a full word image, detects the letters within it from right-to-left, and uses the trained model to 'read' the word.

In [ ]:
def predict_word(image_path, boxes):
    # Sort boxes from Right to Left (Arabic reading order)
    rtl_boxes = sorted(boxes, key=lambda b: -b[0])

    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    batch_images = []

    for (x, y, w, h) in rtl_boxes:
        roi = img[y:y+h, x:x+w]
        if roi.size == 0: continue

        resized = cv2.resize(roi, (32, 32), interpolation=cv2.INTER_AREA)
        batch_images.append(resized.reshape(32, 32, 1).astype('float32') / 255.0)

    if not batch_images: return ""

    # Convert to single numpy batch for efficiency
    input_batch = np.array(batch_images)
    preds = model_arabic_expert.predict(input_batch, verbose=0)
    char_indices = np.argmax(preds, axis=1)

    # AHCD standard mapping
    arabic_alphabet = ['أ', 'ب', 'ت', 'ث', 'ج', 'ح', 'خ', 'د', 'ذ', 'ر', 'ز', 'س', 'ش', 'ص', 'ض', 'ط', 'ظ', 'ع', 'غ', 'ف', 'ق', 'ك', 'ل', 'م', 'ن', 'هـ', 'و', 'ي']
    word_output = [arabic_alphabet[idx] for idx in char_indices]

    return "".join(word_output)

# Run on the first few detections again (optimized)
predicted_string = predict_word('/content/LETTERS.png', final_boxes[:5])
print(f"Optimized Predicted Word Sequence: {predicted_string}")

### 🚀 Auto-Labeling 200k Images
This script uses the `model_arabic_expert` to automatically label your large dataset. It includes a confidence threshold to ensure only high-quality labels are saved.

In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import cv2
import pathlib

def batch_auto_label_recursive(root_folder, confidence_threshold=0.95):
    # AHCD Alphabet Mapping
    arabic_alphabet = ['أ', 'ب', 'ت', 'ث', 'ج', 'ح', 'خ', 'د', 'ذ', 'ر', 'ز', 'س', 'ش', 'ص', 'ض', 'ط', 'ظ', 'ع', 'ر', 'ف', 'ق', 'ك', 'ل', 'م', 'ن', 'هـ', 'و', 'ي']

    # Use pathlib for recursive search across subfolders
    print(f"Scanning {root_folder} for images...")
    extensions = ['*.png', '*.jpg', '*.jpeg', '*.PNG', '*.JPG', '*.JPEG']
    all_files = []
    for ext in extensions:
        all_files.extend(list(pathlib.Path(root_folder).rglob(ext)))

    results = []
    batch_size = 128

    print(f"Processing {len(all_files)} images found in nested folders...")

    for i in tqdm(range(0, len(all_files), batch_size)):
        batch_files = all_files[i : i + batch_size]
        batch_imgs = []
        valid_paths = []

        for f_path in batch_files:
            img = cv2.imread(str(f_path), cv2.IMREAD_GRAYSCALE)
            if img is not None:
                img_resized = cv2.resize(img, (32, 32))
                batch_imgs.append(img_resized.reshape(32, 32, 1).astype('float32') / 255.0)
                valid_paths.append(str(f_path))

        if not batch_imgs: continue

        # Predict batch using the Arabic Expert model
        preds = model_arabic_expert.predict(np.array(batch_imgs), verbose=0)

        for idx, pred in enumerate(preds):
            conf = np.max(pred)
            if conf >= confidence_threshold:
                char_idx = np.argmax(pred)
                results.append({
                    'path': valid_paths[idx],
                    'folder': os.path.basename(os.path.dirname(valid_paths[idx])),
                    'predicted_char': arabic_alphabet[char_idx],
                    'confidence': conf
                })

    # Save to CSV
    df_labels = pd.DataFrame(results)
    df_labels.to_csv('large_scale_auto_labels.csv', index=False)
    print(f"\nDone! Generated {len(results)} labels. Results saved to 'large_scale_auto_labels.csv'.")
    return df_labels

# To run this, ensure your drive is mounted or the local directory is mapped:
# from google.colab import drive
# drive.mount('/content/drive')
# unlabeled_dir = '/content/drive/MyDrive/train/out'
# labels_df = batch_auto_label_recursive(unlabeled_dir)

In [ ]:
from google.colab import drive
# This will prompt for authorization to access your Drive
drive.mount('/content/drive')

In [ ]:
# Define the path to your nested folders
unlabeled_dir = '/content/drive/MyDrive/train/out'

# Run the recursive labeling script
# Results will be saved to 'large_scale_auto_labels.csv'
labels_df = batch_auto_label_recursive(unlabeled_dir)

# Display the first few high-confidence labels found
display(labels_df.head())

In [ ]:
import os
# Diagnostic: List the first 10 items in your drive's train folder to verify the path
try:
    base_path = '/content/drive/MyDrive/train/out'
    if os.path.exists(base_path):
        print(f'Contents of {base_path}:')
        items = os.listdir(base_path)
        print(items[:10])
        if items:
            sub_item = os.path.join(base_path, items[0])
            if os.path.isdir(sub_item):
                print(f'Checking inside first subfolder: {items[0]}')
                print(os.listdir(sub_item)[:10])
    else:
        print(f'The path {base_path} was not found. Checking Drive root...')
        print(os.listdir('/content/drive/MyDrive')[:10])
except Exception as e:
    print(f'Error accessing drive: {e}')

In [ ]:
import os
# Diagnostic: List the first 10 items in your drive's train folder to verify the path
try:
    base_path = '/content/drive/MyDrive/train/out'
    if os.path.exists(base_path):
        print(f"Contents of {base_path}:")
        items = os.listdir(base_path)
        print(items[:10])
        if items:
            sub_item = os.path.join(base_path, items[0])
            if os.path.isdir(sub_item):
                print(f"Checking inside first subfolder: {items[0]}")
                print(os.listdir(sub_item)[:10])
    else:
        print(f"The path {base_path} was not found. Checking Drive root...")
        print(os.listdir('/content/drive/MyDrive')[:10])
except Exception as e:
    print(f"Error accessing drive: {e}")

### 🔍 Final Detection Verification
Review this high-resolution map to ensure every letter has been captured by a red bounding box.

### 🎯 Advanced Detection with RT-DETR
RT-DETR is a real-time end-to-end object detector. We can use it to replace or refine our initial OpenCV-based letter localization.

In [ ]:
# Install Ultralytics to access RT-DETR models
!pip install ultralytics -q

from ultralytics import RTDETR
import cv2
import matplotlib.pyplot as plt

We can load a pre-trained RT-DETR model. For a specialized task like Arabic handwriting, we would eventually fine-tune this on your labeled boxes, but here is the inference setup:

In [ ]:
# Load a pre-trained RT-DETR model (e.g., rtdetr-l)
# Note: For custom letter detection, you would use 'path/to/your_custom_model.pt'
try:
    model_rtdetr = RTDETR('rtdetr-l.pt')
    print("RT-DETR model loaded successfully.")
except Exception as e:
    print(f"Model load skipped or failed: {e}")

def detect_with_rtdetr(image_path):
    results = model_rtdetr(image_path, conf=0.25)
    # Visualize the results
    for r in results:
        im_array = r.plot()  # plot a BGR numpy array of predictions
        plt.figure(figsize=(12, 12))
        plt.imshow(cv2.cvtColor(im_array, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.show()

In [ ]:
import cv2
import matplotlib.pyplot as plt

def verify_detections(image_path, boxes):
    img = cv2.imread(image_path)
    if img is None:
        print('Error: Could not load image')
        return
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Use a very large figure size for clarity
    plt.figure(figsize=(25, 30))
    plt.imshow(img_rgb)

    for i, (x, y, w, h) in enumerate(boxes):
        # Draw Bounding Box
        rect = plt.Rectangle((x, y), w, h, fill=False, edgecolor='red', linewidth=2)
        plt.gca().add_patch(rect)
        # Draw ID number
        plt.text(x, y-5, str(i+1), color='blue', fontsize=12, fontweight='bold',
                 bbox=dict(facecolor='white', alpha=0.5, edgecolor='none', pad=1))

    plt.axis('off')
    plt.title(f'Verification Map: {len(boxes)} Detections Found', fontsize=20)
    plt.show()

verify_detections('/content/LETTERS.png', final_boxes)